# TV2 — Decision Tree (W3-01: Time-based / W3-02: Random Split)

**Role:** Thành viên 2 (Decision Tree Lead)  
**Mục tiêu:** Huấn luyện, tinh chỉnh siêu tham số (Hyperparameter Tuning), trực quan hóa cây quyết định và đánh giá hiệu năng phát hiện tấn công Brute Force FTP/SSH trên tập dữ liệu CICIDS2017 Tuesday.

| Mã công việc | Kịch bản | Phân tách dữ liệu | Không gian đặc trưng | Trọng tâm phân tích |
|---|---|---|---|---|
| **W3-01** | Time-based split | `data/model_ready/time/with_port/` | With Destination Port (67 features) | Khả năng tổng quát hóa theo thời gian & zero-shot transfer |
| **W3-02** | Random split | `data/model_ready/random/with_port/` | With Destination Port (67 features) | Đối chứng hiện tượng rò rỉ dữ liệu (data leakage) |

### Nguyên tắc Protocol nghiêm ngặt:
1. **Chỉ fit trên tập Train**, chọn tham số và ngưỡng tối ưu trên tập **Validation**.
2. **Không chạm vào tập Test** trong quá trình khám phá và tinh chỉnh siêu tham số.
3. Xử lý mất cân bằng lớp bằng `class_weight='balanced'` tính trực tiếp từ Train.
4. Phân tích tính diễn giải của Decision Tree: trực quan hóa cấu trúc rẽ nhánh (`plot_tree`), trích xuất quy tắc phân lớp (`export_text`), và khảo sát hiện tượng quá khớp (Overfitting vs Tree Depth).

In [ ]:
from pathlib import Path
import json, time, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    average_precision_score, roc_auc_score, confusion_matrix,
    precision_recall_curve, roc_curve, classification_report
)

# Xác định thư mục gốc của repository
ROOT = Path.cwd()
if not (ROOT / "data" / "model_ready").exists():
    ROOT = Path.cwd().parent
assert (ROOT / "data" / "model_ready" / "time" / "with_port" / "X_train.csv").exists(), (
    f"Không tìm thấy data/model_ready tại: ROOT={ROOT}"
)

SEED = 42
OUT = ROOT / "artifacts" / "TV2_decision_tree"
OUT.mkdir(parents=True, exist_ok=True)

SCENARIOS = [
    {"split": "time",   "scenario": "with_port", "job": "W3-01"},
    {"split": "random", "scenario": "with_port", "job": "W3-02"},
]

# Lưới siêu tham số cho Decision Tree
GRID = {
    "criterion": ["gini", "entropy"],
    "max_depth": [4, 6, 8, 12, None],
    "min_samples_leaf": [5, 10, 20],
}

print("ROOT =", ROOT)
print("OUT  =", OUT)
print("Tổng số cấu hình thử nghiệm mỗi split:", np.prod([len(v) for v in GRID.values()]))

## 1. Load Dữ liệu & Kiểm tra Phân bố Nhãn

Load tập `Train` và `Validation` từ `data/model_ready/` cho cả 2 kịch bản Time-based và Random split.
- Tập Train theo thời gian chỉ chứa cuộc tấn công `FTP-Patator` (diễn ra buổi sáng).
- Tập Validation chứa cả `FTP-Patator` và một phần `SSH-Patator` (bắt đầu buổi chiều).
- Tập Test theo thời gian chỉ chứa cuộc tấn công `SSH-Patator`.

In [ ]:
def load_xy(split: str, scenario: str, part: str, allow_test: bool = False):
    """Load X, y và Subtype từ data/model_ready."""
    if part == "test" and not allow_test:
        raise RuntimeError("Không được load Test ở giai đoạn tuning/huấn luyện (Protocol Rule).")
    base = ROOT / "data" / "model_ready" / split / scenario
    X = pd.read_csv(base / f"X_{part}.csv")
    y = pd.read_csv(base / f"y_{part}.csv")["BinaryLabel"].astype(int)
    sub = pd.read_csv(base / f"y_{part}_subtype.csv")["Subtype"]
    assert len(X) == len(y) == len(sub), f"Lệch số dòng trong {split}/{scenario}/{part}"
    assert not X.isna().any().any(), f"Phát hiện NaN trong {split}/{scenario}/{part}"
    return X, y, sub

def calculate_metrics(y_true, y_pred, scores=None):
    """Tính toán đầy đủ các chỉ số đánh giá phân loại tấn công."""
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    res = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_attack": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall_attack": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1_attack": float(f1_score(y_true, y_pred, zero_division=0)),
        "fpr": float(fp / (fp + tn)) if (fp + tn) > 0 else 0.0,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "n": int(len(y_true)),
        "n_attack": int(y_true.sum()),
    }
    if scores is not None and len(np.unique(y_true)) == 2:
        res["average_precision"] = float(average_precision_score(y_true, scores))
        res["roc_auc"] = float(roc_auc_score(y_true, scores))
    return res

def get_subtype_recalls(subtypes, y_pred):
    """Đo tỷ lệ phát hiện riêng biệt cho từng loại tấn công FTP vs SSH."""
    y_pred = np.asarray(y_pred, dtype=int)
    subtypes = np.asarray(subtypes)
    out = {}
    for name in ["FTP-Patator", "SSH-Patator"]:
        mask = (subtypes == name)
        out[name] = {
            "support": int(mask.sum()),
            "recall": float(y_pred[mask].mean()) if mask.any() else None
        }
    return out

# Kiểm tra nhanh kích thước dữ liệu
print("=== THỐNG KÊ TẬP DỮ LIỆU HUẤN LUYỆN & KIỂM ĐỊNH ===")
for sc in SCENARIOS:
    Xtr, ytr, str_sub = load_xy(sc["split"], sc["scenario"], "train")
    Xva, yva, sva = load_xy(sc["split"], sc["scenario"], "validation")
    print(f"[{sc['job']} {sc['split']}/{sc['scenario']}]")
    print(f"  Train: {Xtr.shape} | Attack rate: {ytr.mean():.4f} | Subtypes: {str_sub.value_counts().to_dict()}")
    print(f"  Val:   {Xva.shape} | Attack rate: {yva.mean():.4f} | Subtypes: {sva.value_counts().to_dict()}")

## 2. Hyperparameter Grid Search trên Tập Validation

Tiến hành thử nghiệm tổ hợp các siêu tham số `criterion`, `max_depth`, `min_samples_leaf` với `class_weight='balanced'`.
Mục tiêu lựa chọn mô hình: Tối đa hóa **F1 score** trên lớp tấn công (Attack), kèm theo Average Precision cao và FPR thấp nhất.

In [ ]:
def tune_dt(split: str, scenario: str):
    Xtr, ytr, _ = load_xy(split, scenario, "train")
    Xva, yva, sva = load_xy(split, scenario, "validation")
    
    combos = list(itertools.product(GRID["criterion"], GRID["max_depth"], GRID["min_samples_leaf"]))
    rows = []
    
    for i, (crit, depth, leaf) in enumerate(combos, 1):
        t0 = time.time()
        clf = DecisionTreeClassifier(
            criterion=crit,
            max_depth=depth,
            min_samples_leaf=leaf,
            class_weight="balanced",
            random_state=SEED,
        )
        clf.fit(Xtr, ytr)
        fit_time = time.time() - t0
        
        y_val_pred = clf.predict(Xva)
        y_val_scores = clf.predict_proba(Xva)[:, 1]
        
        m = calculate_metrics(yva, y_val_pred, y_val_scores)
        sub_m = get_subtype_recalls(sva, y_val_pred)
        
        m.update({
            "criterion": crit,
            "max_depth": depth if depth is not None else "None",
            "min_samples_leaf": leaf,
            "tree_depth": int(clf.get_depth()),
            "tree_leaves": int(clf.get_n_leaves()),
            "fit_seconds": round(fit_time, 2),
            "split": split,
            "scenario": scenario,
            "ftp_recall": sub_m["FTP-Patator"]["recall"],
            "ssh_recall": sub_m["SSH-Patator"]["recall"],
        })
        rows.append(m)
    
    df = pd.DataFrame(rows).sort_values(["f1_attack", "average_precision"], ascending=False).reset_index(drop=True)
    best = df.iloc[0]
    
    # Refit model tốt nhất trên Train
    best_depth = None if best["max_depth"] == "None" else int(best["max_depth"])
    best_clf = DecisionTreeClassifier(
        criterion=best["criterion"],
        max_depth=best_depth,
        min_samples_leaf=int(best["min_samples_leaf"]),
        class_weight="balanced",
        random_state=SEED,
    )
    best_clf.fit(Xtr, ytr)
    return df, best, best_clf

all_grids = {}
all_best = {}
all_models = {}

for sc in SCENARIOS:
    key = f"{sc['split']}_{sc['scenario']}"
    print(f"\n{'='*30} TUNING: {sc['job']} ({key}) {'='*30}")
    grid_df, best_row, fitted_model = tune_dt(sc["split"], sc["scenario"])
    all_grids[key] = grid_df
    all_best[key] = best_row
    all_models[key] = fitted_model
    
    grid_df.to_csv(OUT / f"dt_grid_{key}.csv", index=False)
    print("Cấu hình tối ưu:")
    print(f"  Criterion: {best_row['criterion']} | Max Depth: {best_row['max_depth']} | Min Samples Leaf: {best_row['min_samples_leaf']}")
    print(f"  Val F1: {best_row['f1_attack']:.4f} | Precision: {best_row['precision_attack']:.4f} | Recall: {best_row['recall_attack']:.4f}")
    print(f"  FTP Recall: {best_row['ftp_recall']} | SSH Recall: {best_row['ssh_recall']}")

## 3. Khảo sát Hiện tượng Quá khớp (Overfitting vs Tree Depth)

Một trong những đặc điểm nổi bật nhất của Decision Tree là tính chất dễ bị quá khớp (overfitting) khi độ sâu của cây tăng quá lớn.
Đồ thị dưới đây so sánh điểm F1 trên tập Train và tập Validation khi `max_depth` thay đổi từ 2 đến 20 trên kịch bản Time-based.

In [ ]:
Xtr_time, ytr_time, _ = load_xy("time", "with_port", "train")
Xva_time, yva_time, _ = load_xy("time", "with_port", "validation")

depth_range = list(range(2, 21))
train_f1s = []
val_f1s = []

for d in depth_range:
    tree_exp = DecisionTreeClassifier(max_depth=d, min_samples_leaf=5, class_weight="balanced", random_state=SEED)
    tree_exp.fit(Xtr_time, ytr_time)
    tr_pred = tree_exp.predict(Xtr_time)
    va_pred = tree_exp.predict(Xva_time)
    train_f1s.append(f1_score(ytr_time, tr_pred))
    val_f1s.append(f1_score(yva_time, va_pred))

plt.figure(figsize=(10, 5))
plt.plot(depth_range, train_f1s, 'o-', label='Train F1 Score', color='blue')
plt.plot(depth_range, val_f1s, 's--', label='Validation F1 Score', color='red')
plt.axvline(x=6, color='green', linestyle=':', label='Chosen Depth (d=6)')
plt.xlabel('Tree Max Depth (max_depth)', fontsize=11)
plt.ylabel('F1 Score (Attack)', fontsize=11)
plt.title('Phân tích Overfitting: Train vs Validation F1 theo Độ sâu Cây (Time-based)', fontsize=13, fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=10)
plt.tight_layout()
plt.savefig(OUT / "overfitting_depth_analysis.png", dpi=180)
plt.show()

print("Nhận xét: Ở max_depth = 6-8, mô hình đạt điểm tối ưu trên Validation. Khi tăng chiều sâu lên > 12, Train F1 đạt xấp xỉ 1.0 trong khi Validation F1 không tăng thêm hoặc suy giảm, biểu hiện đặc trưng của overfit.")

## 4. Tính Diễn giải (Interpretability): Trực quan hóa & Trích xuất Quy tắc

Decision Tree là mô hình hộp trắng (white-box model). Dưới đây là:
1. **Feature Importances**: Các đặc trưng quyết định phân loại hàng đầu.
2. **Cấu trúc rẽ nhánh** (`plot_tree`) của 3 tầng đầu tiên.
3. **Quy tắc phân loại** (`export_text`) dưới dạng mã giả logic.

In [ ]:
model_time = all_models["time_with_port"]
feature_names = list(Xtr_time.columns)

# 1. Feature Importances
feat_imp = pd.DataFrame({
    "feature": feature_names,
    "importance": model_time.feature_importances_
}).sort_values("importance", ascending=False).reset_index(drop=True)
feat_imp.to_csv(OUT / "dt_feature_importances.csv", index=False)

plt.figure(figsize=(10, 6))
top_feats = feat_imp.head(15).iloc[::-1]
plt.barh(top_feats["feature"], top_feats["importance"], color="#2ca02c")
plt.xlabel("Gini Feature Importance", fontsize=11)
plt.title("Top 15 Đặc trưng Quan trọng Nhất - Decision Tree (W3-01)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT / "dt_feature_importance_top15.png", dpi=180)
plt.show()

# 2. Trực quan hóa cấu trúc rẽ nhánh
fig, ax = plt.subplots(figsize=(22, 9))
plot_tree(
    model_time,
    max_depth=3,
    feature_names=feature_names,
    class_names=["BENIGN", "Attack"],
    filled=True,
    rounded=True,
    fontsize=8,
    ax=ax
)
plt.title("Sơ đồ Cấu trúc Cây Quyết định (3 Tầng Đầu) - W3-01 Time-based", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT / "tree_structure_preview.png", dpi=200)
plt.show()

# 3. Trích xuất quy tắc phân lớp
rules = export_text(model_time, feature_names=feature_names, max_depth=3)
print("=== QUY TẮC RẼ NHÁNH TIÊU BIỂU (TOP 3 TẦNG) ===")
print(rules[:1200])
with open(OUT / "tree_rules.txt", "w", encoding="utf-8") as f:
    f.write(rules)

## 5. Đánh giá trên Tập Test (W3-01 & W3-02)

Sau khi mô hình và các siêu tham số đã được cố định hoàn toàn từ tập Validation, ta mở tập **Test** để đánh giá khả năng tổng quát hóa thực tế.

### Câu hỏi kiểm chứng:
- Trên **W3-01 (Time-based)**: Test set chỉ chứa cuộc tấn công `SSH-Patator` trong khi Train chỉ chứa `FTP-Patator`. Mô hình Decision Tree đơn lẻ có chuyển giao (zero-shot transfer) để phát hiện SSH được không?
- Trên **W3-02 (Random split)**: Do các flow của cả 2 loại tấn công được trộn đều giữa Train và Test, liệu kết quả có đạt F1 cao vượt trội phản ánh hiện tượng rò rỉ thông tin?

In [ ]:
test_summary = []

for sc in SCENARIOS:
    key = f"{sc['split']}_{sc['scenario']}"
    model = all_models[key]
    Xte, yte, ste = load_xy(sc["split"], sc["scenario"], "test", allow_test=True)
    
    scores = model.predict_proba(Xte)[:, 1]
    pred = model.predict(Xte)
    
    m = calculate_metrics(yte, pred, scores)
    sub_m = get_subtype_recalls(ste, pred)
    
    row = {
        "job": sc["job"],
        "split": sc["split"],
        "scenario": sc["scenario"],
        "criterion": all_best[key]["criterion"],
        "max_depth": all_best[key]["max_depth"],
        "min_samples_leaf": all_best[key]["min_samples_leaf"],
        "accuracy": round(m["accuracy"], 4),
        "precision": round(m["precision_attack"], 4),
        "recall": round(m["recall_attack"], 4),
        "f1_score": round(m["f1_attack"], 4),
        "fpr": round(m["fpr"], 6),
        "roc_auc": round(m.get("roc_auc", 0.0), 4),
        "avg_precision": round(m.get("average_precision", 0.0), 4),
        "ftp_recall": sub_m["FTP-Patator"]["recall"],
        "ssh_recall": sub_m["SSH-Patator"]["recall"],
    }
    test_summary.append(row)
    
    print(f"\n===== KẾT QUẢ TEST: {sc['job']} ({key}) =====")
    print(classification_report(yte, pred, target_names=["BENIGN", "Attack"], digits=4))

test_df = pd.DataFrame(test_summary)
test_df.to_csv(OUT / "dt_test_results.csv", index=False)
test_df

## 6. Tổng kết & Bàn giao Artifacts TV2

Lưu các model file dạng `.joblib` và JSON metadata đầy đủ vào `artifacts/TV2_decision_tree/` để phục vụ tổng hợp đồ án chung của nhóm.

In [ ]:
import joblib

# Lưu các mô hình đã huấn luyện
for key, model in all_models.items():
    joblib.dump(model, OUT / f"dt_{key}.joblib")

# Lưu metadata tham số chính thức
final_params = {
    "author": "TV2 (Decision Tree Lead)",
    "models": {
        "W3-01": test_summary[0],
        "W3-02": test_summary[1],
    },
    "note": (
        "W3-01 Time-based thể hiện rõ thách thức zero-shot transfer khi Test set chỉ gồm SSH-Patator "
        "trong khi Train chỉ có FTP-Patator. W3-02 Random split minh chứng điểm F1 cao do rò rỉ thời gian."
    )
}
with open(OUT / "dt_final_params.json", "w", encoding="utf-8") as f:
    json.dump(final_params, f, indent=2, ensure_ascii=False)

print("Đã lưu toàn bộ artifacts TV2 tại:", OUT)
print("Hoàn tất notebook thực nghiệm cho TV2!")